# Bài 8: Các phép biến đổi hình học (Geometric Transformations)

Bài này trình bày cách di chuyển, thay đổi kích thước và biến dạng ảnh: lật, cắt, xoay và co giãn, sau đó là hệ phân cấp tổng quát hơn của các phép biến đổi 2D &mdash; **Euclid**, **đồng dạng (similarity)** và **affine** &mdash; mà các thao tác trên đều là trường hợp đặc biệt.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## Một ảnh thử nghiệm không có tính đối xứng

Ta vẽ chữ "F" vì nó không có đối xứng xoay hay đối xứng gương &mdash; mỗi phép lật và xoay đều tạo ra kết quả khác biệt rõ ràng, giúp dễ phân biệt các phép biến đổi.

In [ ]:
def make_f_image(size=160):
    img = np.zeros((size, size, 3), dtype=np.uint8)
    img[:] = (30, 30, 30)
    color = (255, 200, 0)
    cv2.rectangle(img, (40, 20), (65, 140), color, -1)   # vertical stroke
    cv2.rectangle(img, (40, 20), (120, 45), color, -1)   # top horizontal stroke
    cv2.rectangle(img, (40, 65), (100, 90), color, -1)   # middle horizontal stroke
    return img

img = make_f_image()
plt.imshow(img)
plt.title('Original')
plt.axis('off')
plt.show()

## Các thao tác cơ bản

### Lật

`cv2.flip` lật gương một ảnh: mã `1` lật ngang (trái-phải), `0` lật dọc (trên-dưới), `-1` lật cả hai.

In [ ]:
flip_h = cv2.flip(img, 1)
flip_v = cv2.flip(img, 0)
flip_hv = cv2.flip(img, -1)

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax, im, title in zip(axes, [img, flip_h, flip_v, flip_hv],
                          ['Original', 'flip horizontal', 'flip vertical', 'flip both']):
    ax.imshow(im)
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

### Cắt

Cắt ảnh chỉ là cắt lát mảng &mdash; không cần hàm OpenCV nào. `image[y0:y1, x0:x1]` giữ lại các hàng `y0..y1` và các cột `x0..x1`.

In [ ]:
cropped = img[10:100, 30:110]

plt.imshow(cropped)
plt.title(f'Cropped, shape={cropped.shape[:2]}')
plt.axis('off')
plt.show()

### Co giãn (thay đổi kích thước)

`cv2.resize` thay đổi kích thước ảnh, có thể dùng hệ số khác nhau cho chiều rộng và chiều cao. Tỉ số giữa chiều rộng và chiều cao là **tỉ lệ khung hình (aspect ratio)**. Co giãn đều giữ nguyên hình dạng (tỉ lệ khung hình), còn co giãn không đều sẽ kéo dài hoặc ép dẹt nó.

In [ ]:
uniform = cv2.resize(img, None, fx=0.5, fy=0.5)
stretched = cv2.resize(img, None, fx=1.5, fy=0.6)

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
for ax, im, title in zip(axes, [img, uniform, stretched],
                          ['Original', 'Uniform scale (0.5, 0.5)', 'Non-uniform (1.5, 0.6)']):
    ax.imshow(im)
    ax.set_title(f'{title}\nshape={im.shape[:2]}', fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

### Xoay

`cv2.getRotationMatrix2D` tạo một ma trận $2\times3$ để xoay một góc tùy ý quanh một tâm được chọn, sau đó `cv2.warpAffine` áp dụng ma trận này lên ảnh.

In [ ]:
h, w = img.shape[:2]
center = (w / 2, h / 2)
M = cv2.getRotationMatrix2D(center, angle=25, scale=1.0)
print('rotation matrix:\n', M)

rotated = cv2.warpAffine(img, M, (w, h))

plt.imshow(rotated)
plt.title('Rotated 25 degrees about the center')
plt.axis('off')
plt.show()

## Hệ phân cấp các phép biến đổi

Lật, xoay, co giãn và tịnh tiến đều là các trường hợp của một phép biến đổi ma trận $2\times3$ tổng quát

$$\begin{bmatrix}x'\\y'\end{bmatrix} = \begin{bmatrix}a & b\\c & d\end{bmatrix}\begin{bmatrix}x\\y\end{bmatrix} + \begin{bmatrix}t_x\\t_y\end{bmatrix}$$

được áp dụng bằng `cv2.warpAffine`. Ràng buộc phần $2\times2$ theo các cách khác nhau cho ra một họ phép biến đổi lồng nhau, từ chặt nhất đến lỏng nhất:

| Phép biến đổi | Tham số tự do | Dạng ma trận | Bảo toàn |
|---|---|---|---|
| **Euclid (cứng — rigid)** | góc xoay $\theta$, tịnh tiến $(t_x,t_y)$ | $\begin{bmatrix}\cos\theta & -\sin\theta\\ \sin\theta & \cos\theta\end{bmatrix}$ | độ dài, góc |
| **Đồng dạng** | + hệ số co giãn đều $s$ | $s\begin{bmatrix}\cos\theta & -\sin\theta\\ \sin\theta & \cos\theta\end{bmatrix}$ | góc, tỉ số độ dài |
| **Affine** | mọi ma trận $2\times2$ khả nghịch | $\begin{bmatrix}a & b\\c & d\end{bmatrix}$ | tính song song, tỉ số dọc theo một đường thẳng |

Có hai thứ thay đổi khi đi xuống bảng, theo hai chiều ngược nhau. *Tập các phép biến đổi được phép* lớn dần: mọi phép biến đổi Euclid đều là trường hợp đặc biệt của phép đồng dạng (chỉ cần $s=1$), và mọi phép đồng dạng đều là trường hợp đặc biệt của phép affine (chỉ là ma trận xoay-và-co-giãn thay vì một ma trận tùy ý) &mdash; nên các phép biến đổi ở mỗi hàng là tập cha của hàng phía trên. Đổi lại cho sự linh hoạt thêm đó, *các tính chất được đảm bảo giữ nguyên* lại thu hẹp: phép Euclid bảo toàn cả độ dài lẫn góc; phép đồng dạng từ bỏ việc bảo toàn bản thân độ dài (chỉ còn tỉ số giữa chúng); phép affine từ bỏ luôn cả góc. Phép affine là tổng quát nhất trong ba loại: nó có thể làm trượt (shear) một hình vuông thành hình bình hành, điều mà cả phép Euclid lẫn phép đồng dạng đều không làm được.

### Thấy sự khác biệt trên một hình vuông đơn vị

Cách rõ nhất để thấy mỗi phép biến đổi làm được gì và không làm được gì là quan sát điều xảy ra với một hình vuông đơn giản.

In [ ]:
square = np.array([[0, 0], [1, 0], [1, 1], [0, 1], [0, 0]], dtype=np.float64)

def apply_2x2(pts, A, t=(0, 0)):
    return pts @ A.T + np.array(t)

theta = np.radians(30)
R = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])

euclidean_sq  = apply_2x2(square, R, t=(0.3, 0.1))
similarity_sq = apply_2x2(square, 1.6 * R, t=(0.3, 0.1))
affine_sq     = apply_2x2(square, np.array([[1.6, 0.7], [0.2, 0.9]]), t=(0.3, 0.1))

fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
for ax, pts, title in zip(
    axes,
    [euclidean_sq, similarity_sq, affine_sq],
    ['Euclidean\n(rotate + translate)', 'Similarity\n(+ uniform scale)', 'Affine\n(shear allowed)'],
):
    ax.plot(*square.T, '--', color='gray', label='original')
    ax.plot(*pts.T, color='#e74c3c', linewidth=2, label='transformed')
    ax.set_aspect('equal')
    ax.set_title(title, fontsize=9)
    ax.legend(fontsize=7, loc='upper left')
plt.tight_layout()
plt.show()

Chỉ có hình vuông affine là không còn trông giống một hình vuông (có thể đã đổi kích thước) &mdash; các góc của nó không còn là 90 độ, vì phép affine cho phép trượt. Cả ba đều giữ các cạnh đối song song. (Các phép biến đổi xạ ảnh (projective) tổng quát hơn, ví dụ phối cảnh của camera, sẽ được trình bày ở Bài 24.)

Ảnh động này cho thấy hình vuông biến đổi liên tục tới phép biến đổi cuối cùng rồi quay lại. Hình vuông Euclid giữ cứng, hình vuông đồng dạng lớn lên nhưng vẫn "có dạng vuông", còn hình vuông affine trượt các góc vuông của nó thành hình bình hành.

<img src="../animations/transform_hierarchy_demo.gif" alt="Ảnh động biến đổi một hình vuông đơn vị từ phép đồng nhất sang phép Euclid, đồng dạng và affine, cho thấy tính chất nào (độ dài, góc, tính song song) được giữ lại ở mỗi phép" style="display: block; margin: 0 auto; max-width: 900px;">

### Ước lượng phép biến đổi affine từ các cặp điểm tương ứng

Trên thực tế, ta thường không biết trực tiếp ma trận biến đổi. Ví dụ: một vật phẳng được chụp *nghiêng* (hoặc một bản quét bị lệch), và ta muốn hoàn tác sự biến dạng. Điều này đòi hỏi xác định một vài điểm mốc (landmark) trong cả ảnh bị lệch lẫn vị trí mà chúng *đáng lẽ* phải ở trong một góc nhìn chuẩn, chính diện. `cv2.getAffineTransform` nhận 3 cặp điểm tương ứng như vậy (số lượng tối thiểu để xác định cả 6 tham số affine) và trả về ma trận ánh xạ tập này lên tập kia — giải theo chiều từ-ảnh-lệch-về-chuẩn, rồi áp dụng nó sẽ *nắn thẳng (rectify)* bức ảnh.

In [ ]:
canvas_size = (w + 60, h + 60)

# simulate a photo taken at an angle: apply a KNOWN affine warp (rotation + shear + scale)
# -- in a real application this warp is unknown; we only ever get to see `warped` below
theta = np.radians(20)
R = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
shear = np.array([[1, 0.25], [0, 1]])
A_true = R @ shear * 1.1
t_true = np.array([40, 20])
M_true = np.hstack([A_true, t_true.reshape(2, 1)])

warped = cv2.warpAffine(img, M_true, canvas_size)

# 3 landmarks we can identify in BOTH images (e.g. by clicking on the F's outer corners)
canonical_pts = np.float32([[40, 20], [120, 20], [40, 140]])
warped_pts = ((A_true @ canonical_pts.T).T + t_true).astype(np.float32)  # where those 3 points ended up

# solve for the warped -> canonical transform, then apply it to rectify the whole photo
M_recovered = cv2.getAffineTransform(warped_pts, canonical_pts)
rectified = cv2.warpAffine(warped, M_recovered, (w, h))

pixel_error = np.abs(rectified.astype(int) - img.astype(int)).mean()
print('mean abs pixel error, rectified vs. true original:', round(pixel_error, 2))

fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))
axes[0].imshow(img)
axes[0].set_title('Original (canonical)', fontsize=9)
axes[1].imshow(warped)
for p in warped_pts:
    axes[1].scatter(*p, c='red', s=30)
axes[1].set_title('Warped photo\n(3 identified landmarks)', fontsize=9)
axes[2].imshow(rectified)
axes[2].set_title('Rectified\n(unwarped back)', fontsize=9)
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

Ảnh động này cho thấy phép biến dạng làm méo bức ảnh, sau đó phép biến đổi khôi phục được sẽ hoàn tác nó, đồng thời chỉ ra 3 điểm mốc trong suốt quá trình. Bảng bên cạnh nêu bật một sự bất đối xứng quan trọng: giai đoạn 1 áp dụng trực tiếp một ma trận (thường là chưa biết), còn giai đoạn 2 trước hết phải *giải* ra ma trận đó từ 3 cặp điểm tương ứng rồi mới áp dụng được.

<img src="../animations/affine_rectify_demo.gif" alt="Ảnh động chuỗi biến dạng-rồi-nắn-thẳng, cho thấy một bức ảnh bị làm méo bởi một phép affine đã biết rồi được khôi phục nhờ 3 điểm mốc được theo dõi, kèm bảng bên giải thích sự bất đối xứng giữa việc áp dụng một phép biến dạng đã biết và việc giải ra một phép biến dạng chưa biết" style="display: block; margin: 0 auto; max-width: 680px;">

### Bài tập

1. `cv2.flip` không thuộc họ affine mà ta đã viết dưới dạng ma trận ở trên (ma trận xoay $R$ luôn có định thức $+1$). Ma trận $2\times2$ của một phép lật có định thức bằng bao nhiêu? Hãy dựng ma trận $2\times2$ cho `flip(1)` và kiểm tra.
2. Dựng một ma trận biến đổi đồng dạng với $s=1$ và xác nhận nó trùng với một phép biến đổi Euclid thuần túy &mdash; tức phép đồng dạng là sự tổng quát hóa thực sự của phép Euclid, không phải một họ khác.
3. Sửa ma trận affine trong ví dụ hình vuông đơn vị sao cho các cạnh đối *không* song song. Điều gì phải bị phá vỡ để chuyện đó xảy ra? (Gợi ý: đây chính là ranh giới phân tách phép affine với phép xạ ảnh.)